# 店舗売上需要予測: 実験 #012 3大トップモデルの対数空間加重ブレンド

- **コンペ名:** [Store Sales - Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting)
- **目的:** 
  1. これまで構築した最高峰モデル群の多様性（アンサンブル・ダイバーシティ）を対数空間で最大統合
  2. Exp #010（本番最高: LB 0.38968 / 76位）を主軸（60%）に採用
  3. Exp #011（手元検証最高: Val 0.39765 / 2シード平均）を補完（30%）に配置
  4. Exp #008（初代再帰自己回帰: LB 0.39084 / 82位）をアンカー（10%）に配合
  5. 直近21日間完全ゼロ売上実績の124系列に対する厳格なゼロポストプロセッシングの適用
- **評価指標:** **RMSLE (Root Mean Squared Logarithmic Error)**
- **結果:** **Public LB `0.38908` (71位 / 542チーム中 自己ベスト更新！)**
  - Top 10%（メダル圏、Rank 54: 0.38602）まで残り **0.00306** / 17人抜き圏内へ到達！

## 1. ライブラリ読み込み & データ準備

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# パス設定
base_dir = '../'
sub10_path = os.path.join(base_dir, 'submission_exp010.csv')
sub11_path = os.path.join(base_dir, 'submission_exp011.csv')
sub8_path = os.path.join(base_dir, 'submission_exp008.csv')
train_path = os.path.join(base_dir, 'data', 'train.csv')
test_path = os.path.join(base_dir, 'data', 'test.csv')

## 2. トップモデル予測値の読み込みと相関確認

In [ ]:
sub10 = pd.read_csv(sub10_path)
sub11 = pd.read_csv(sub11_path)
sub8 = pd.read_csv(sub8_path)

print(f"Exp #010 平均売上: {sub10['sales'].mean():.2f}")
print(f"Exp #011 平均売上: {sub11['sales'].mean():.2f}")
print(f"Exp #008 平均売上: {sub8['sales'].mean():.2f}")

pred_df = pd.DataFrame({
    'Exp010': sub10['sales'],
    'Exp011': sub11['sales'],
    'Exp008': sub8['sales']
})
print("\n--- 予測相関行列 ---")
print(pred_df.corr())

## 3. 対数空間加重ブレンド (Log-space Weighted Blending)

RMSLE 評価指標では、通常の算術平均ではなく $\log(y + 1)$ 空間での加重平均を行うことが幾何平均効果をもたらし、極端なスパイク誤差を劇的に抑制します。

In [ ]:
# 重み設定: 60% : 30% : 10%
w10, w11, w8 = 0.60, 0.30, 0.10

log10 = np.log1p(np.clip(sub10['sales'].values, 0, None))
log11 = np.log1p(np.clip(sub11['sales'].values, 0, None))
log8 = np.log1p(np.clip(sub8['sales'].values, 0, None))

log_blend = w10 * log10 + w11 * log11 + w8 * log8
sales_blend = np.clip(np.expm1(log_blend), 0, None)

print(f"対数ブレンド後 平均売上: {np.mean(sales_blend):.2f}")

## 4. 直近21日ゼロ売上フィルター（124系列）の厳格適用

直近21日間（2017-07-25〜2017-08-15）に1度も売上が立っていない商品・店舗のペアは、取り扱い停止または店舗改修中とみなし、予測を完全ゼロにクリップします。

In [ ]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
train_df['date'] = pd.to_datetime(train_df['date'])

recent_train = train_df[train_df['date'] >= '2017-07-25']
recent_sum = recent_train.groupby(['store_nbr', 'family'])['sales'].sum().reset_index()
zero_pairs = set(zip(recent_sum[recent_sum['sales'] == 0]['store_nbr'], recent_sum[recent_sum['sales'] == 0]['family']))
print(f"直近21日完全ゼロ系列数: {len(zero_pairs)} 系列 (テスト期間 16日 x 124 = {len(zero_pairs)*16} 行)")

sub_df = pd.DataFrame({'id': sub10['id'], 'sales': sales_blend})
test_merged = pd.merge(test_df[['id', 'store_nbr', 'family']], sub_df, on='id')

zero_mask = [((st, fam) in zero_pairs) for st, fam in zip(test_merged['store_nbr'], test_merged['family'])]
sales_blend[zero_mask] = 0.0

print(f"ゼロクリップ後のゼロ予測件数: {(sales_blend == 0).sum()} 件")
print(f"最終平均売上: {np.mean(sales_blend):.2f}")

## 5. 提出ファイルの生成と確認

In [ ]:
final_sub = pd.DataFrame({
    'id': sub10['id'],
    'sales': sales_blend
}).sort_values('id').reset_index(drop=True)

out_path = os.path.join(base_dir, 'submission_exp012.csv')
final_sub.to_csv(out_path, index=False)
print(f"提出ファイル保存完了: {out_path} ({len(final_sub)} 行)")
print(final_sub.head(10))